In [ ]:
%load_ext blackcellmagic 
# %black -l 120
%load_ext autoreload
%autoreload 2

In [ ]:

config_down_scaled_prev = {
    "n_actions": 4,
    "batch_size": 16,
    "architectures": ["cnn"], #[,"impala"]
    "feature_list": [[16, 24, 24]],
    "gap_list": [True],
    "layer_norm": (True, True),
    "low_scale": True,
    "n_conv": 3,
    "n_fc": 1,
}

config_large_scale = {
    "n_actions": 4,
    "batch_size": 32,
    "architectures": ["cnn"], #[,"impala"]
    "feature_list": [[32, 64, 64, 512]],
    "gap_list": [False],
    "layer_norm": (False, False),
    "low_scale": False,
    "n_conv": 3,
    "n_fc": 2,
}




In [ ]:
import os
#reproducability and determinism
os.environ["XLA_FLAGS"] = (
    #"--xla_gpu_autotune_level=0 "
    #"--xla_gpu_deterministic_ops=true "
    "--xla_backend_optimization_level=0 "
)

from slimdqn.algorithms.dqn import DQN
from slimdqn.algorithms.dqnrcshared import DQNRCShared
from slimdqn.algorithms.idqnshared import iDQNShared
from slimdqn.algorithms.gidqnshared import GiDQNShared


import jax
import jax.numpy as jnp
from tests.utils import Generator

def run(n_actions, batch_size, architectures, feature_list, gap_list, layer_norm, low_scale, n_conv, n_fc):
    def count_params(params):
        return sum(x.size for x in jax.tree.leaves(params))


    def count_flops(q, has_target_params=False):
        best_action_compiled = (
            jax.jit(q.best_action).lower(q.params, sample_generator.state(jax.random.PRNGKey(0))).compile()
        )
        if not has_target_params:
            learn_on_batch_compiled = (
                jax.jit(q.learn_on_batch)
                .lower(q.params, q.optimizer_state, sample_generator.samples(jax.random.PRNGKey(0)), jnp.ones(batch_size))
                .compile()
            )
        else:
            learn_on_batch_compiled = (
                jax.jit(q.learn_on_batch)
                .lower(
                    q.params,
                    q.target_params,
                    q.optimizer_state,
                    sample_generator.samples(jax.random.PRNGKey(0)),
                    jnp.ones(batch_size),
                )
                .compile()
            )

        return best_action_compiled, learn_on_batch_compiled

    pixels_frame_stack= (42, 42, 2) if low_scale else (84, 84, 4) #pixel x pixel, frame stack
    sample_generator = Generator(batch_size, pixels_frame_stack, n_actions)


    metrics = {}
    metrics["flops"] = {}
    metrics["num_params"] = {}

    for idx, architecture in enumerate(architectures):
        features = feature_list[idx]
        gap = gap_list[idx]
        print(f"--- DQN - {architecture} ---")
        q_dqn = DQN(
            jax.random.PRNGKey(0),
            observation_dim= pixels_frame_stack,
            n_actions= n_actions,
            features= features,
            architecture_type= architecture,
            layer_norm = layer_norm,
            gap=gap,
            learning_rate= 6.25e-5,
            gamma= 0.99,
            update_horizon= 1,
            update_to_data= 0.25,
            target_update_period= 8000,
            low_scale=low_scale,
            n_conv= n_conv ,
            n_fc=n_fc,
        )
        metrics["num_params"][f"dqn_{architecture}"] = count_params(q_dqn.params) + count_params(q_dqn.target_params)
        q_dqn_best_action_compiled, q_dqn_learn_on_batch_compiled = count_flops(q_dqn, has_target_params=True)
        metrics["flops"][f"dqn_{architecture}"] = q_dqn_learn_on_batch_compiled.cost_analysis()[0]["flops"]
        print("DQN Num params: ", metrics["num_params"][f"dqn_{architecture}"])
        print("DQN FLOPs best action: ", q_dqn_best_action_compiled.cost_analysis()[0]["flops"])
        print("DQN FLOPs to learn on a batch: ", metrics["flops"][f"dqn_{architecture}"], "\n")


        print("Linear Flops both:", q_dqn_best_action_compiled.cost_analysis()[0]["flops"] + metrics["flops"][f"dqn_{architecture}"], "\n")

        # print(f"--- DQNRC - {architecture} ---")
        # q_qrc = DQNRCShared(
        #     jax.random.PRNGKey(0),
        #     observation_dim= pixels_frame_stack,
        #     n_actions= n_actions,
        #     features= features,
        #     architecture_type= architecture,
        #     layer_norm = layer_norm,
        #     gap=gap,
        #     linear_heads=True,
        #     learning_rate= 6.25e-5,
        #     gamma= 0.99,
        #     update_horizon= 1,
        #     update_to_data= 0.25,
        #     target_update_period= 8000,
        #     weight_decay= 1,
        #     low_scale=low_scale,
        #     n_conv= n_conv ,
        #     n_fc=n_fc,
        # )
        # metrics["num_params"][f"qrc_{architecture}"] = count_params(q_qrc.params)
        # q_qrc_best_action_compiled, q_qrc_learn_on_batch_compiled = count_flops(q_qrc, has_target_params=False)
        # metrics["flops"][f"qrc_{architecture}"] = q_qrc_learn_on_batch_compiled.cost_analysis()[0]["flops"]
        # print("DQNRC with linear heads params: ", metrics["num_params"][f"qrc_{architecture}"])
        # print("DQNRC FLOPs best action: ", q_qrc_best_action_compiled.cost_analysis()[0]["flops"])
        # print("DQNRC FLOPs to learn on a batch: ", metrics["flops"][f"qrc_{architecture}"], "\n")
        #
        # print("Linear Flops both:", q_qrc_best_action_compiled.cost_analysis()[0]["flops"] + metrics["flops"][f"qrc_{architecture}"], "\n")
        #
        # print(f"--- i-DQN - {architecture} ---")
        # q_idqn = iDQNShared(
        #     jax.random.PRNGKey(0),
        #     observation_dim= pixels_frame_stack,
        #     n_actions= n_actions,
        #     n_bellman_iterations=5,
        #     features= features,
        #     architecture_type= architecture,
        #     layer_norm = layer_norm,
        #     gap=gap,
        #     linear_heads=True,
        #     learning_rate= 6.25e-5,
        #     gamma= 0.99,
        #     update_horizon= 1,
        #     update_to_data= 0.25,
        #     target_update_period= 8000,
        #     low_scale=low_scale,
        #     n_conv= n_conv ,
        #     n_fc=n_fc,
        # )
        # metrics["num_params"][f"idqn_{architecture}"] = count_params(q_idqn.params) + count_params(q_idqn.target_params)
        # q_idqn_best_action_compiled, q_idqn_learn_on_batch_compiled = count_flops(q_idqn, has_target_params=True)
        # metrics["flops"][f"idqn_{architecture}"] = q_idqn_learn_on_batch_compiled.cost_analysis()[0]["flops"]
        # print("iDQN with linear heads params: ", metrics["num_params"][f"idqn_{architecture}"])
        # print("Linear i-DQN FLOPs best action: ", q_idqn_best_action_compiled.cost_analysis()[0]["flops"])
        # print("Linear i-DQN FLOPs to learn on a batch: ", metrics["flops"][f"idqn_{architecture}"], "\n")
        #
        # print("Linear Flops both:", q_idqn_best_action_compiled.cost_analysis()[0]["flops"] + metrics["flops"][f"idqn_{architecture}"], "\n")
        #
        # print(f"--- Gi-DQN - {architecture} ---")
        # q_gidqn = GiDQNShared(
        #     jax.random.PRNGKey(0),
        #     observation_dim= pixels_frame_stack,
        #     n_actions= n_actions,
        #     n_bellman_iterations=5,
        #     features= features,
        #     architecture_type= architecture,
        #     layer_norm = layer_norm,
        #     gap=gap,
        #     linear_heads=True,
        #     learning_rate= 6.25e-5,
        #     gamma= 0.99,
        #     update_horizon= 1,
        #     update_to_data= 0.25,
        #     target_update_period= 8000,
        #     weight_decay= 1,
        #     low_scale=low_scale,
        #     n_conv= n_conv ,
        #     n_fc=n_fc,
        # )
        # metrics["num_params"][f"gidqn_{architecture}"] = count_params(q_gidqn.params) + count_params(q_gidqn.target_params)
        # q_gidqn_best_action_compiled, q_gidqn_learn_on_batch_compiled = count_flops(q_gidqn, has_target_params=True)
        # metrics["flops"][f"gidqn_{architecture}"] = q_gidqn_learn_on_batch_compiled.cost_analysis()[0]["flops"]
        # print("GiDQN Num params with linear heads: ", metrics["num_params"][f"gidqn_{architecture}"])
        # print("Linear FLOPs best action: ", q_gidqn_best_action_compiled.cost_analysis()[0]["flops"])
        # print("Linear FLOPs to learn on a batch: ", metrics["flops"][f"gidqn_{architecture}"], "\n")
        # print("Linear Flops both:", q_gidqn_best_action_compiled.cost_analysis()[0]["flops"] + metrics["flops"][f"gidqn_{architecture}"], "\n")

    print(metrics)

In [ ]:
print("--------Large Scale -----------------------")
run(**config_large_scale)

# print("---------- Down Scaled ----------------------")
# run(**config_down_scaled)


print("---------- Down Scaled LN ----------------------")
run(**config_down_scaled_LN)